# OASIS-2 Coronal 2-Qubit Analysis

Dashboard notebook for the classical CNN vs 2-qubit hybrid CQ-CNN experiment. The `.py` script remains the reproducible training engine; this notebook reads CSV outputs, plots behavior, and stores interpretation notes.

In [ ]:
from pathlib import Path
import subprocess
import sys

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style="whitegrid", context="notebook")

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "classification":
    PROJECT_ROOT = PROJECT_ROOT.parents[1]
elif PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

EXPERIMENT_SCRIPT = PROJECT_ROOT / "experiments/oasis2/oasis2_coronal_experiment.py"
RESULT_DIR = PROJECT_ROOT / "results/classification/oasis2_coronal_128_64"
CLASSICAL_CSV = RESULT_DIR / "oasis2_coronal_2qubit_classical.csv"
HYBRID_CSV = RESULT_DIR / "oasis2_coronal_2qubit_hybrid.csv"

PROJECT_ROOT, EXPERIMENT_SCRIPT, RESULT_DIR

## Optional: refresh experiment

Set `RUN_EXPERIMENT = True` only when you want the notebook to rerun training and overwrite the current CSVs.

In [ ]:
RUN_EXPERIMENT = False

if RUN_EXPERIMENT:
    command = [sys.executable, str(EXPERIMENT_SCRIPT)]
    completed = subprocess.run(command, cwd=PROJECT_ROOT, text=True, capture_output=True)
    print(completed.stdout)
    if completed.returncode != 0:
        print(completed.stderr)
        raise RuntimeError(f"Experiment failed with exit code {completed.returncode}")

## Load CSV outputs

In [ ]:
missing = [path for path in [CLASSICAL_CSV, HYBRID_CSV] if not path.exists()]
if missing:
    raise FileNotFoundError(f"Missing result CSVs: {missing}. Run the experiment first.")

classical = pd.read_csv(CLASSICAL_CSV)
hybrid = pd.read_csv(HYBRID_CSV)
results = pd.concat([classical, hybrid], ignore_index=True)

epoch_df = results[results["row_type"] == "epoch"].copy()
final_df = results[results["row_type"] == "final"].copy()
summary_df = results[results["row_type"] == "summary"].copy()

numeric_cols = ["epoch", "train_loss", "train_acc", "test_loss", "test_acc", "test_macro_f1", "test_balanced_acc", "macro_f1", "balanced_acc", "auc", "sensitivity", "specificity"]
for frame in [epoch_df, final_df, summary_df]:
    for col in numeric_cols:
        if col in frame.columns:
            frame[col] = pd.to_numeric(frame[col], errors="coerce")

print(f"Epoch rows: {len(epoch_df)}")
print(f"Final rows: {len(final_df)}")
print(f"Summary rows: {len(summary_df)}")
final_df.head()

## Summary metrics

In [ ]:
summary_df[["model", "eval_split", "test_loss", "test_acc", "macro_f1", "balanced_acc", "auc", "sensitivity", "specificity"]]

## Learning curves

In [ ]:
plot_metrics = ["train_loss", "test_loss", "train_acc", "test_acc", "test_macro_f1", "test_balanced_acc"]
fig, axes = plt.subplots(2, 3, figsize=(15, 8), constrained_layout=True)
for ax, metric in zip(axes.flatten(), plot_metrics):
    sns.lineplot(data=epoch_df, x="epoch", y=metric, hue="model", style="trial", markers=True, dashes=False, ax=ax)
    ax.set_title(metric)
plt.show()

## Final trial comparison

In [ ]:
original_final = final_df[final_df["eval_split"] == "original"].copy()
display_cols = ["model", "trial", "seed", "test_loss", "test_acc", "macro_f1", "balanced_acc", "auc", "tn", "fp", "fn", "tp"]
original_final[display_cols].sort_values(["trial", "model"])

In [ ]:
metrics = ["test_acc", "macro_f1", "balanced_acc", "auc"]
long_final = original_final.melt(id_vars=["model", "trial", "seed"], value_vars=metrics, var_name="metric", value_name="value")
plt.figure(figsize=(12, 5))
sns.barplot(data=long_final, x="metric", y="value", hue="model", errorbar="sd")
plt.ylim(0, 1.05)
plt.title("Final Metrics Across Trials")
plt.show()

## Winner by primary metric

In [ ]:
winner_table = (
    original_final.groupby("model")[["macro_f1", "balanced_acc", "auc", "test_loss"]]
    .mean()
    .sort_values(["macro_f1", "balanced_acc", "auc", "test_loss"], ascending=[False, False, False, True])
)
winner_table

## Notes / Interpretation

- What changed in this run?
- Which model won on average?
- Did the hybrid model converge consistently or only for some seeds?
- Did macro F1 and balanced accuracy agree with plain accuracy?
- What should be tried next?